In [ ]:
%matplotlib inline

import os
import csv
import copy
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

INPUT_PATH = "related_files/cor_all.txt"
OUTPUT_PNG_PATH = "related_files/cor_all_widget.png"
ANNOTATIONS_SAVE_CSV = "related_files/highlight_annotations_adjusted.csv"

HIGHLIGHT_GENES = {

    "aligned_OG0000235_pruned":"FKH2",
    "aligned_OG0000632_pruned_1":"TRA1",
    "aligned_OG0000626_pruned_1":"TOR1",

    "aligned_OG0000711_pruned":"SSD1",
    "aligned_OG0002274_pruned":"CDC3",
    "aligned_OG0001808_pruned":"SEC31 (WD40)",
    "aligned_OG0001132_pruned":"VPS15 (Pkinase)",
    "aligned_OG0003258_pruned":"CreC (WD40)",
    "aligned_OG0001752_pruned":"APM4",
    "aligned_OG0003664_pruned":"MSN2 (zf-C2H2)",
    "aligned_OG0004547_pruned":"TPO4 (MFS_1)",
    "aligned_OG0003150_pruned":"PUB1 (RRM_1)",
    "aligned_OG0000349_pruned":"ARO80 (Zn_clus)",
}

COLOR_NEG = "#b2478b"
COLOR_POS = "#353c8a"
COLOR_GRAY = "gray"
RHO_THRESHOLD = 0.35
P_THRESHOLD = 0.05

FIGSIZE = (8, 6)
SAVE_DPI = 600

df = pd.read_csv(INPUT_PATH, sep="\t")

required_cols = {"gene", "Rho", "P"}
if not required_cols.issubset(set(df.columns)):
    raise RuntimeError(f"Input file must contain columns: {required_cols}. Found: {list(df.columns)}")

df["Rho"] = df["Rho"].astype(float)
df["P"] = df["P"].astype(float)

gene_xy_map = {}
for idx, row in df.iterrows():
    gene = row["gene"]
    if gene in HIGHLIGHT_GENES:
        gene_xy_map[gene] = (float(row["Rho"]), float(row["P"]))

if len(gene_xy_map) == 0:
    raise RuntimeError("No highlight genes found in the input file. Check HIGHLIGHT_GENES and the 'gene' column in the table.")

annotation_state = {}
for gene_id, label in HIGHLIGHT_GENES.items():
    if gene_id in gene_xy_map:
        tx, ty = gene_xy_map[gene_id]

        offset_x = 0.02
        offset_y = 0.02
        annotation_state[gene_id] = {
            "label": label,
            "target_x": tx,
            "target_y": ty,
            "offset_x": offset_x,
            "offset_y": offset_y
        }

def draw_figure(state, show_in_output=True):

    plt.close('all')
    fig, ax = plt.subplots(figsize=FIGSIZE)

    rhos = df["Rho"].values
    p_values = df["P"].values

    sig_mask = (abs(rhos) > RHO_THRESHOLD) & (p_values < P_THRESHOLD)
    neg_mask = sig_mask & (rhos < 0)
    pos_mask = sig_mask & (rhos > 0)
    nonsig_mask = ~sig_mask

    ax.scatter(rhos[nonsig_mask], p_values[nonsig_mask], edgecolor=COLOR_GRAY, facecolor="none", s=10, alpha=0.6, linewidths=0.5)

    if neg_mask.any():
        ax.scatter(rhos[neg_mask], p_values[neg_mask], edgecolor=COLOR_NEG, facecolor="none", s=20, alpha=0.8, linewidths=0.6)
    if pos_mask.any():
        ax.scatter(rhos[pos_mask], p_values[pos_mask], edgecolor=COLOR_POS, facecolor="none", s=20, alpha=0.8, linewidths=0.6)

    for gene_id, info in state.items():
        tx = info["target_x"]
        ty = info["target_y"]
        label = info["label"]
        ox = info["offset_x"]
        oy = info["offset_y"]
        text_x = tx + ox
        text_y = ty + oy

        point_color = COLOR_NEG if tx < 0 else COLOR_POS
        ax.scatter([tx], [ty], color=point_color, s=60, zorder=3)
        ax.annotate(
            label,
            xy=(tx, ty),
            xytext=(text_x, text_y),
            textcoords="data",
            fontsize=10,
            color="black",
            arrowprops=dict(arrowstyle="->", color="gray", lw=0.8, shrinkA=2, shrinkB=2)
        )

    ax.axvline(RHO_THRESHOLD, color="black", linestyle="--", linewidth=1)
    ax.axvline(-RHO_THRESHOLD, color="black", linestyle="--", linewidth=1)
    ax.set_yscale("log")
    ax.invert_yaxis()
    yticks = [1, 0.1, 0.01, 0.001, 0.0001]
    ax.set_yticks(yticks)
    ax.set_yticklabels([str(y) for y in yticks])
    ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_xlabel("Rho", fontsize=12)
    ax.set_ylabel("p-value", fontsize=12)
    plt.tight_layout()
    if show_in_output:
        display(fig)
    return fig, ax

out_box = widgets.Output()
with out_box:
    draw_figure(annotation_state)

gene_dropdown = widgets.Dropdown(
    options=list(annotation_state.keys()),
    description='Gene:',
    disabled=False
)

first_gene = next(iter(annotation_state))
label_text = widgets.Text(
    value=annotation_state[first_gene]["label"],
    description='Label:',
    disabled=False
)

offset_x_box = widgets.FloatText(
    value=annotation_state[first_gene]["offset_x"],
    description='Offset X:',
    step=0.01
)
offset_y_box = widgets.FloatText(
    value=annotation_state[first_gene]["offset_y"],
    description='Offset Y:',
    step=0.01
)

apply_button = widgets.Button(description="Apply", button_style="primary")
reset_button = widgets.Button(description="Reset to defaults")
save_csv_button = widgets.Button(description="Save annotations (CSV)")
save_png_button = widgets.Button(description="Save PNG")

status = widgets.Output()

controls = widgets.VBox([
    widgets.HBox([gene_dropdown, label_text]),
    widgets.HBox([offset_x_box, offset_y_box]),
    widgets.HBox([apply_button, reset_button, save_csv_button, save_png_button]),
    status
])

display(controls, out_box)

def on_gene_change(change):

    gene = change['new']
    if gene is None:
        return
    info = annotation_state[gene]
    label_text.value = info['label']
    offset_x_box.value = info['offset_x']
    offset_y_box.value = info['offset_y']

gene_dropdown.observe(on_gene_change, names='value')

def on_apply_clicked(b):

    gene = gene_dropdown.value
    if gene is None:
        with status:
            clear_output(wait=True)
            print("No gene selected.")
        return

    annotation_state[gene]["label"] = label_text.value.strip() if label_text.value.strip() != "" else annotation_state[gene]["label"]
    try:
        annotation_state[gene]["offset_x"] = float(offset_x_box.value)
        annotation_state[gene]["offset_y"] = float(offset_y_box.value)
    except Exception:
        with status:
            clear_output(wait=True)
            print("Invalid offset values. Please enter numeric values.")
        return

    with out_box:
        clear_output(wait=True)
        draw_figure(annotation_state)
    with status:
        clear_output(wait=True)
        print(f"Applied changes to {gene}: label='{annotation_state[gene]['label']}', offset_x={annotation_state[gene]['offset_x']}, offset_y={annotation_state[gene]['offset_y']}")

apply_button.on_click(on_apply_clicked)

def on_reset_clicked(b):

    for gene_id in list(annotation_state.keys()):
        tx = annotation_state[gene_id]["target_x"]
        annotation_state[gene_id]["label"] = HIGHLIGHT_GENES.get(gene_id, annotation_state[gene_id]["label"])
        annotation_state[gene_id]["offset_x"] = 0.02
        annotation_state[gene_id]["offset_y"] = 0.02

    gene = gene_dropdown.value
    if gene is not None:
        label_text.value = annotation_state[gene]["label"]
        offset_x_box.value = annotation_state[gene]["offset_x"]
        offset_y_box.value = annotation_state[gene]["offset_y"]
    with out_box:
        clear_output(wait=True)
        draw_figure(annotation_state)
    with status:
        clear_output(wait=True)
        print("All annotations reset to default positions and labels.")

reset_button.on_click(on_reset_clicked)

def on_save_csv_clicked(b):

    parent = os.path.dirname(ANNOTATIONS_SAVE_CSV)
    if parent and not os.path.exists(parent):
        os.makedirs(parent, exist_ok=True)
    fieldnames = ["gene_id", "displayed_text", "target_x", "target_y", "text_x", "text_y"]
    rows = []
    for gid, info in annotation_state.items():
        rows.append({
            "gene_id": gid,
            "displayed_text": info["label"],
            "target_x": info["target_x"],
            "target_y": info["target_y"],
            "text_x": info["target_x"] + info["offset_x"],
            "text_y": info["target_y"] + info["offset_y"]
        })
    try:
        with open(ANNOTATIONS_SAVE_CSV, "w", newline="") as fh:
            writer = csv.DictWriter(fh, fieldnames=fieldnames)
            writer.writeheader()
            for r in rows:
                writer.writerow(r)
        with status:
            clear_output(wait=True)
            print(f"Annotations saved to {ANNOTATIONS_SAVE_CSV}")
    except Exception as e:
        with status:
            clear_output(wait=True)
            print(f"Failed to save annotations: {e}")

save_csv_button.on_click(on_save_csv_clicked)

def on_save_png_clicked(b):

    try:

        fig, _ = draw_figure(annotation_state, show_in_output=False)
        fig.savefig(OUTPUT_PNG_PATH, dpi=SAVE_DPI, format="png")
        plt.close(fig)
        with status:
            clear_output(wait=True)
            print(f"Current figure saved to {OUTPUT_PNG_PATH}")
    except Exception as e:
        with status:
            clear_output(wait=True)
            print(f"Failed to save figure: {e}")

save_png_button.on_click(on_save_png_clicked)
